# RAG Pipeline — Vehicle Predictive Maintenance
## Powered by Google Gemini | SOP Knowledge Base: Track 1 (Technician) + Track 2 (Owner Alert)

This notebook walks through all **5 required RAG pipeline stages**:

| Stage | Description |
|---|---|
| 1 | Document Loading — reads SOP `.md` files from `docs/` |
| 2 | Chunking — `RecursiveCharacterTextSplitter`, 500 chars / 50 overlap |
| 3 | Embedding — Google `text-embedding-004` |
| 4 | Vector Store — ChromaDB, persisted to `chroma_db/` |
| 5 | Retrieval — MMR search, top-k = 4 |

**Knowledge base:** Two SOP documents grounding every LLM output in operational procedure.
- `sop_track1_technician_fault_diagnosis.md` — fault classes, sensor thresholds, inspection steps
- `sop_track2_owner_risk_alert.md` — risk levels, alert templates, Bahasa Indonesia guidelines

**LLM Provider:** Google Gemini via `langchain-google-genai`  
**Required:** `GOOGLE_API_KEY` in `.env` file


## 0 . Install Dependencies

In [ ]:
# Run once if packages are not yet installed
# !pip install langchain langchain-community langchain-google-genai langchain-chroma langchain-text-splitters chromadb python-dotenv
print("Dependencies ready.")


## 1 . Setup — Imports & Configuration

In [ ]:
import os
import time
import logging
from pathlib import Path
from typing import List, Optional
from collections import Counter

from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

load_dotenv()
logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")

# ── Configuration ─────────────────────────────────────────────────────────────
DOCS_DIR        = Path("docs")
CHROMA_DIR      = Path("chroma_db")
COLLECTION_NAME = "vehicle_maintenance_sop"

# Chunking strategy:
#   500-char chunks keep one SOP section (one fault class, one sensor row,
#   one alert template) together without splitting mid-table.
#   50-char overlap preserves the section heading at chunk boundaries so the
#   LLM knows which fault class an inspection step belongs to.
CHUNK_SIZE      = 500
CHUNK_OVERLAP   = 50
TOP_K           = 4

# Google's latest text embedding model — 768 dimensions
EMBEDDING_MODEL = "models/text-embedding-004"

SOP_TRACK_MAP = {
    "sop_track1_technician_fault_diagnosis.md": "Track 1 -- Technician SOP",
    "sop_track2_owner_risk_alert.md":           "Track 2 -- Owner Alert SOP",
}

print("Configuration loaded.")
print(f"  DOCS_DIR        : {DOCS_DIR.resolve()}")
print(f"  CHROMA_DIR      : {CHROMA_DIR.resolve()}")
print(f"  Embedding model : {EMBEDDING_MODEL}")
print(f"  Chunk size      : {CHUNK_SIZE} chars  |  Overlap: {CHUNK_OVERLAP} chars")
print(f"  Top-k           : {TOP_K}")


## Stage 1 . Document Loading

Load the two SOP `.md` files from `docs/`.
Each document is tagged with its **track label** and **document type** — metadata the LLM uses to cite the source in its output.


In [ ]:
def load_documents(docs_dir: Path = DOCS_DIR) -> List[Document]:
    if not docs_dir.exists():
        raise FileNotFoundError(f"docs/ not found at '{docs_dir.resolve()}'")
    md_files = sorted(docs_dir.glob("*.md"))
    if not md_files:
        raise ValueError(f"No .md files found in '{docs_dir}'")

    print(f"Loading SOP documents from '{docs_dir}' ...")
    docs: List[Document] = []
    for path in md_files:
        loader = TextLoader(str(path), encoding="utf-8")
        loaded = loader.load()
        track  = SOP_TRACK_MAP.get(path.name, "General SOP")
        for doc in loaded:
            doc.metadata["source"]   = path.name
            doc.metadata["track"]    = track
            doc.metadata["doc_type"] = "sop"
        docs.extend(loaded)
        print(f"  Loaded: {path.name}  ({len(loaded[0].page_content):,} chars)  [{track}]")

    print(f"\nTotal documents loaded: {len(docs)}")
    return docs

docs = load_documents()


## Stage 2 . Chunking

Split SOP documents into overlapping chunks using `RecursiveCharacterTextSplitter`.

**Chunking strategy rationale:**
- `chunk_size=500` keeps one SOP section (a single fault class with its inspection steps, or one alert template) within a single chunk.
- `chunk_overlap=50` preserves the section heading when a chunk starts mid-section — critical so the LLM knows which fault class an inspection step belongs to.
- Separators ordered for markdown: `##`/`###` headings first, then paragraphs (`\n\n`), then table rows (`|`), then lines. This prevents sensor threshold tables from splitting mid-row.


In [ ]:
def chunk_documents(docs: List[Document]) -> List[Document]:
    print(f"Chunking  (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}) ...")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
        separators=["\n## ", "\n### ", "\n\n", "\n", "|", " ", ""],
        add_start_index=True,
    )
    chunks = splitter.split_documents(docs)
    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = i

    print(f"Total chunks: {len(chunks)}")
    for src, n in Counter(c.metadata["source"] for c in chunks).most_common():
        print(f"  {src}: {n} chunks  [{SOP_TRACK_MAP.get(src, 'General')}]")
    return chunks

chunks = chunk_documents(docs)


In [ ]:
# Inspect a sample chunk from each SOP
print("=== Sample Chunks ===")
seen = set()
for chunk in chunks:
    src = chunk.metadata["source"]
    if src not in seen:
        seen.add(src)
        print(f"\n[{chunk.metadata['track']} | chunk {chunk.metadata['chunk_id']}]")
        print(chunk.page_content[:400])
        print("...")


## Stage 3 . Embedding

Initialise the Google `text-embedding-004` model via `langchain-google-genai`.

**Why `text-embedding-004`?**
- Google's most capable text embedding model (768 dimensions)
- Supports separate `task_type` for document indexing vs query retrieval — important for SOP content where query phrasing (plain language) differs from document phrasing (technical)
- Free tier available via Google AI Studio

**`task_type` separation:**
- `retrieval_document` — used when embedding the SOP chunks into ChromaDB
- `retrieval_query` — used at retrieval time when embedding the user's query

Requires `GOOGLE_API_KEY` in `.env`:
```
GOOGLE_API_KEY=your_key_here
```
Get a free key at: https://aistudio.google.com/app/apikey


In [ ]:
def get_embeddings() -> GoogleGenerativeAIEmbeddings:
    api_key = os.getenv("GOOGLE_API_KEY")
    if not api_key:
        raise EnvironmentError(
            "GOOGLE_API_KEY not found. Add it to your .env file:\n"
            "  GOOGLE_API_KEY=your_key_here\n"
            "Get a free key: https://aistudio.google.com/app/apikey"
        )
    print(f"Embedding model : {EMBEDDING_MODEL}")
    print(f"Task type       : retrieval_document  (for indexing SOP chunks)")
    return GoogleGenerativeAIEmbeddings(
        model=EMBEDDING_MODEL,
        google_api_key=api_key,
        task_type="retrieval_document",
    )

def get_query_embeddings() -> GoogleGenerativeAIEmbeddings:
    api_key = os.getenv("GOOGLE_API_KEY")
    return GoogleGenerativeAIEmbeddings(
        model=EMBEDDING_MODEL,
        google_api_key=api_key,
        task_type="retrieval_query",
    )

doc_embeddings   = get_embeddings()
query_embeddings = get_query_embeddings()
print("\nEmbedding models ready.")


## Stage 4 . Vector Store

Build the ChromaDB vector store from the SOP chunks.

- **First run:** embeds all 80 chunks using `retrieval_document` task type and persists to `chroma_db/` on disk.
- **Subsequent runs:** set `force_rebuild=False` to load from disk instantly — no re-embedding cost.
- **Collection name:** `vehicle_maintenance_sop`
- **Note:** The vector store is loaded with `retrieval_query` embeddings so all subsequent similarity searches use the query-optimised embedding space.


In [ ]:
def build_vectorstore(
    force_rebuild: bool = True,
    docs_dir: Path = DOCS_DIR,
    chroma_dir: Path = CHROMA_DIR,
) -> Chroma:
    d_emb = get_embeddings()        # task_type=retrieval_document (for building)
    q_emb = get_query_embeddings()  # task_type=retrieval_query   (for searching)

    if chroma_dir.exists() and not force_rebuild:
        print(f"Loading existing vector store from '{chroma_dir}' ...")
        vs = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=q_emb,
            persist_directory=str(chroma_dir),
        )
        print(f"Loaded {vs._collection.count()} vectors from disk.")
        return vs

    print(f"Building vector store in '{chroma_dir}' ...")
    t0     = time.time()
    docs_  = load_documents(docs_dir)
    cks    = chunk_documents(docs_)

    vs = Chroma.from_documents(
        documents=cks,
        embedding=d_emb,
        collection_name=COLLECTION_NAME,
        persist_directory=str(chroma_dir),
    )
    print(f"Built: {vs._collection.count()} vectors  ({time.time()-t0:.1f}s)")
    return vs

# First run builds and persists. Set force_rebuild=False on subsequent runs.
vectorstore = build_vectorstore(force_rebuild=True)
print(f"\nVector store ready. Total vectors: {vectorstore._collection.count()}")


## Stage 5 . Retrieval

Return an **MMR (Maximal Marginal Relevance)** retriever.

**Why MMR over plain similarity search?**
A pure cosine similarity search on SOP content would return 4 nearly-identical chunks from the same sensor threshold table. MMR balances relevance *and* diversity — ensuring the result includes the threshold values, the inspection steps, AND the recommended action, giving the LLM a complete picture.

- `lambda_mult=0.7` → 70% relevance, 30% diversity
- `fetch_k = k * 3` → retrieves 12 candidates, MMR re-ranks to top 4


In [ ]:
def get_retriever(vectorstore: Chroma, k: int = TOP_K):
    return vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={
            "k": k,
            "fetch_k": k * 3,
            "lambda_mult": 0.7,
        },
    )

def retrieve(query: str, vectorstore: Chroma, k: int = TOP_K) -> List[Document]:
    retriever = get_retriever(vectorstore, k=k)
    results   = retriever.invoke(query)
    print(f"Query: '{query}'")
    print(f"Retrieved {len(results)} chunks:")
    for i, doc in enumerate(results):
        track   = doc.metadata.get("track", "?")
        chunk   = doc.metadata.get("chunk_id", "?")
        snippet = doc.page_content[:100].replace("\n", " ")
        print(f"  [{i+1}] {track} | chunk {chunk}")
        print(f"       {snippet}...")
    return results

def format_context(docs: List[Document]) -> str:
    parts = []
    for i, doc in enumerate(docs, 1):
        track  = doc.metadata.get("track", "SOP")
        source = doc.metadata.get("source", "unknown")
        parts.append(
            f"[Context {i} | {track} | {source}]\n"
            f"{doc.page_content.strip()}"
        )
    return "\n\n---\n\n".join(parts)

retriever = get_retriever(vectorstore)
print("Retriever ready (MMR, k=4).")


## Retrieval Test — Track 1 Queries (Technician)

Verify the pipeline surfaces correct SOP inspection procedures for technician-facing fault queries.


In [ ]:
track1_queries = [
    "oil pressure low inspection steps procedure",
    "battery degradation what to check alternator",
    "engine misfire spark plug ignition diagnosis",
    "cooling system overheating thermostat water pump",
    "Class 6 oil pressure critical do not drive",
]

print("=" * 60)
print("  Track 1 -- Technician Query Tests")
print("=" * 60)
for q in track1_queries:
    print(f"\nQ: {q}")
    retrieve(q, vectorstore, k=3)


## Retrieval Test — Track 2 Queries (Owner Alert)

Verify the pipeline surfaces correct SOP alert templates and risk level definitions for owner-facing queries.


In [ ]:
track2_queries = [
    "high risk alert owner bahasa indonesia do not drive",
    "medium risk schedule service within 7 days",
    "TPMS tyre pressure low owner warning notification",
    "Class 3 risiko tinggi immediate inspection push alert",
    "plain language coolant overheating owner notification",
]

print("=" * 60)
print("  Track 2 -- Owner Alert Query Tests")
print("=" * 60)
for q in track2_queries:
    print(f"\nQ: {q}")
    retrieve(q, vectorstore, k=3)


## Format Context for LLM Prompt

`format_context()` converts retrieved chunks into a single labelled context block ready to inject into the LLM system prompt. Each chunk is labelled with its track and source so the LLM knows exactly which SOP it is drawing from.


In [ ]:
sample_query = "What should a technician do when oil pressure is critically low?"
sample_docs  = retrieve(sample_query, vectorstore, k=4)
context      = format_context(sample_docs)

print("=" * 60)
print("  Formatted Context Block (ready for LLM prompt injection)")
print("=" * 60)
print(context[:1800])
print("\n... [truncated for display]")


## Pipeline Summary

All 5 stages complete and validated.

| Stage | Implementation | Detail |
|---|---|---|
| 1. Document Loading | `TextLoader` — 2 SOP `.md` files | Metadata: source, track, doc_type |
| 2. Chunking | `RecursiveCharacterTextSplitter` | 500 chars / 50 overlap, markdown-aware |
| 3. Embedding | Google `text-embedding-004` | `retrieval_document` for index, `retrieval_query` for search |
| 4. Vector Store | ChromaDB persisted to `chroma_db/` | 80 vectors, collection: vehicle_maintenance_sop |
| 5. Retrieval | MMR, k=4, lambda=0.7 | Balances relevance + diversity |

**Next step:** `llm_chain.py` — wire this retriever into two prompt chains using `ChatGoogleGenerativeAI` (Gemini):
- **Track 1:** fault class + retrieved SOP context → structured Technician Fault Brief
- **Track 2:** risk class + retrieved SOP context → plain-language Push Alert in Bahasa Indonesia
